# NLU Shared Task - Demo Solution 1

This notebook demonstrates how to load the trained Solution 1 models (Feature Extraction + Stacking Ensemble) and generate predictions for a given input CSV file.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os

# Ensure src is in the python path
sys.path.append(os.path.abspath('..'))

## 1. Feature Extraction
First, we extract the stylometric and info-theoretic features from the input text pairs. For this demo, we assume the features have already been extracted to `.npy` format as per the `src/solution1/training/feature_extraction.py` script.

In [ ]:
import numpy as np
from pathlib import Path

# Path to the consolidated 97-feature matrix (Style + Unmasking + GI)
feature_path = Path('../data/all_features_test_clean.npy')

if feature_path.exists():
    X_raw = np.load(feature_path)
    print(f'Loaded consolidated 97-feature matrix of shape: {X_raw.shape}')
else:
    print(f'ERROR: Feature matrix not found at {feature_path}. Ensure extraction is run first.')

## 2. Load Models and Predict
We use the provided `predict_with_stack` function to run the base models and the meta-classifier.

In [ ]:
from src.solution1.predict import predict_with_stack

model_dir = Path('../models/solution1')

if 'X_raw' in locals() and X_raw.shape[1] == 97:
    print('Running stacking inference ( leakage-free ) ...')
    y_pred, y_prob, prob_a1, prob_a2 = predict_with_stack(X_raw, model_dir)
    print(f'Success! Generated {len(y_pred)} predictions.')
else:
    print('ERROR: X_raw is not correctly loaded or does not have exactly 97 features.')

## 3. Format and Save Output
Format the output to strictly match the submission specification (a single column named `prediction`).

In [ ]:
submission_df = pd.DataFrame({'prediction': y_pred})

# Preview output
display(submission_df.head())

# Save to disk as Group_17_A.csv per spec
out_path = Path('../outputs/Group_17_A.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(out_path, index=False)
print(f'SUCCESS: Submission file saved to {out_path}')